<a href="https://colab.research.google.com/github/dora-lai/fyp-aspect-financial-news/blob/main/Report_Data_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import sys
# Clear partially loaded nltk modules from memory cache
for key in list(sys.modules.keys()):
    if key.startswith('nltk'):
        del sys.modules[key]

import nltk
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pysentiment2 as ps


# 1. LLM Performance on FiQA
try:
    df_fiqa = pd.read_csv("fiqa_llm_predictions.csv")
    p, r, f1, _ = precision_recall_fscore_support(df_fiqa['true_label'], df_fiqa['llm_pred'], average='macro', zero_division=0)
    print(f"1. LLM FiQA Macro-F1: {f1:.4f}")
except FileNotFoundError:
    print("1. File not found: fiqa_llm_predictions.csv")

# 2. LM Performance on Gold Set
try:
    df_gold = pd.read_csv("llm_run_1_merged.csv")

    # Remove sentiment and empty sentence row
    df_gold = df_gold.dropna(subset=['sentiment', 'sentence']).copy()
    df_gold['sentiment'] = df_gold['sentiment'].astype(str).str.strip().str.lower()
    df_gold['sentence'] = df_gold['sentence'].astype(str)

    lm = ps.LM()
    def eval_lm(text):
        tokens = lm.tokenize(text)
        score = lm.get_score(tokens).get('Polarity', 0.0)
        return 'positive' if score > 0.1 else ('negative' if score < -0.1 else 'neutral')

    df_gold['lm_pred'] = df_gold['sentence'].apply(eval_lm)

    lm_acc = accuracy_score(df_gold['sentiment'], df_gold['lm_pred'])
    _, _, lm_f1, _ = precision_recall_fscore_support(df_gold['sentiment'], df_gold['lm_pred'], average='macro', zero_division=0)
    print(f"2. LM US Gold Macro-F1: {lm_f1:.4f}")

    # 3. Gold Set Entity and Pair accuracy
    for c in ['entity', 'aspect', 'llm_entity', 'llm_aspect', 'llm_sentiment']:
        df_gold[c] = df_gold[c].astype(str).str.strip().str.lower()

    entity_acc = (df_gold['entity'] == df_gold['llm_entity']).mean()
    pair_acc = ((df_gold['aspect'] == df_gold['llm_aspect']) &
                (df_gold['sentiment'] == df_gold['llm_sentiment'])).mean()

    print(f"3. Gold Set Entity Accuracy: {entity_acc:.4f}")
    print(f"4. Gold Set Pair Exact Match: {pair_acc:.4f}")

except FileNotFoundError:
    print("2. File not found: llm_run_1_merged.csv")

1. LLM FiQA Macro-F1: 0.5604
2. LM US Gold Macro-F1: 0.4460
3. Gold Set Entity Accuracy: 0.7639
4. Gold Set Pair Exact Match: 0.5694


In [7]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# load FiQA prediction
df_fiqa = pd.read_csv("fiqa_llm_predictions.csv")

# calculate
precision, recall, f1, _ = precision_recall_fscore_support(
    df_fiqa['true_label'],
    df_fiqa['llm_pred'],
    labels=['positive', 'neutral', 'negative'],
    average=None
)

print("LLM on FiQA:")
print(f"Precision (Macro): {sum(precision)/3:.4f}")
print(f"Recall (Macro):    {sum(recall)/3:.4f}")
print(f"F1 (pos): {f1[0]:.4f} | F1 (neu): {f1[1]:.4f} | F1 (neg): {f1[2]:.4f}")

LLM on FiQA:
Precision (Macro): 0.7506
Recall (Macro):    0.7467
F1 (pos): 0.7619 | F1 (neu): 0.6596 | F1 (neg): 0.8200
